# Airport Baggage Inspection Queue Simulation

A reproducible next-event simulation for exploring staffing and waiting-time tradeoffs at a secondary inspection station. **Synthetic baseline only** until the airport dataset is obtained and its timestamps are audited.

**Provenance:** AI-generated draft; human review pending.

## 1. Problem and question
How does the number of inspection stations affect average queue wait, the probability of waiting longer than five minutes, and utilisation? This notebook does **not** assume that screening event timestamps are arrivals to the overall checkpoint.

## 2. Model, assumptions and equations
One infinite-capacity FIFO queue feeding $c$ identical parallel stations. Baseline arrivals form a Poisson process with rate $\lambda$, and independent exponential service times have mean $1/\mu$ (M/M/$c$). Events are arrivals and departures. The system is stable in steady state when $\rho=\lambda/(c\mu)<1$. A future empirical-service model samples durations from the audited source data, yielding an M/G/$c$ simulation; M/M/$c$ formulae are **not** an analytic reference for that empirical model.

For the M/M/$c$ benchmark, $a=\lambda/\mu$, $P_0=[\sum_{n=0}^{c-1}a^n/n!+a^c/(c!(1-\rho))]^{-1}$, $P_W=a^c P_0/(c!(1-\rho))$, and $W_q=P_W/(c\mu-\lambda)$.

**Units:** all time quantities in minutes. **Provenance:** AI-generated draft; human review pending.

## 3. Implementation

In [ ]:
# Standalone Colab cell; Python queue engine is identical to src/queue_model.py.
# Source provenance: ChatGPT-generated, human review required.
import math, statistics, csv, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as student_t

"""Single FIFO queue with parallel identical inspection stations; units: minutes."""
from collections import deque
from dataclasses import dataclass
from math import inf
import random

@dataclass
class Result:
    arrivals: int
    completed: int
    mean_wait: float
    fraction_wait_over_target: float
    mean_queue_length: float
    utilisation: float
    waits: list

def simulate(arrivals, services, servers=2, target_wait=5.0):
    """Deterministic event-trace engine; arrivals and services are equal-length sequences."""
    if servers < 1 or len(arrivals) != len(services):
        raise ValueError("Invalid server count or trace lengths")
    if any(b < a for a,b in zip(arrivals,arrivals[1:])):
        raise ValueError("Arrivals must be sorted")
    if any(s <= 0 for s in services):
        raise ValueError("Service durations must be positive")
    n=len(arrivals)
    if not n:
        return Result(0,0,0,0,0,0,[])
    idle=list(range(servers))
    finish=[inf]*servers
    queue=deque()
    waits=[0.0]*n
    i=0; done=0; now=0.; queue_area=0.; busy_area=0.
    horizon_start=arrivals[0]
    while done<n:
        next_arrival=arrivals[i] if i<n else inf
        next_depart=min(finish)
        t=min(next_arrival,next_depart)
        queue_area+=len(queue)*(t-now)
        busy_area+=sum(f<inf for f in finish)*(t-now)
        now=t
        # Departures before arrivals at tied timestamps.
        departing=[j for j,f in enumerate(finish) if f==now]
        for j in departing:
            finish[j]=inf; done+=1; idle.append(j)
        while i<n and arrivals[i]==now:
            queue.append(i); i+=1
        idle.sort()
        while idle and queue:
            j=idle.pop(0)
            customer=queue.popleft()
            waits[customer]=now-arrivals[customer]
            finish[j]=now+services[customer]
    duration=now-horizon_start
    return Result(n,done,sum(waits)/n,sum(w>target_wait for w in waits)/n,
                  queue_area/duration if duration else 0,
                  busy_area/(servers*duration) if duration else 0,waits)

def random_trace(rate_per_minute, mean_service_minutes, n, seed):
    if rate_per_minute<=0 or mean_service_minutes<=0: raise ValueError("Rates must be positive")
    rng=random.Random(seed); t=0.; arrivals=[]; services=[]
    for _ in range(n):
        t+=rng.expovariate(rate_per_minute)
        arrivals.append(t)
        services.append(rng.expovariate(1/mean_service_minutes))
    return arrivals,services



In [ ]:
def mmc_wait(lambda_rate, mu_rate, servers):
    if not 0 < lambda_rate < servers * mu_rate:
        raise ValueError("Requires stable M/M/c parameters")
    a = lambda_rate / mu_rate
    rho = a / servers
    p0 = 1 / (sum(a**k/math.factorial(k) for k in range(servers)) + a**servers/(math.factorial(servers)*(1-rho)))
    pw = a**servers * p0/(math.factorial(servers)*(1-rho))
    return pw/(servers*mu_rate-lambda_rate)

def replication(rate, mean_service, servers, n, seed, warmup=1000):
    arrivals, services = random_trace(rate, mean_service, n + warmup, seed)
    result = simulate(arrivals, services, servers=servers)
    # Warm-up is discarded from customer waiting observations only.
    # Area/time statistics require a separate steady-state window implementation.
    return float(np.mean(result.waits[warmup:])), float(np.mean(np.asarray(result.waits[warmup:]) > 5))

def mean_ci(values, alpha=0.05):
    values = np.asarray(values, dtype=float)
    if len(values) < 2: raise ValueError("At least two replications required")
    avg = float(values.mean())
    half = float(student_t.ppf(1-alpha/2, len(values)-1) * values.std(ddof=1)/np.sqrt(len(values)))
    return avg, avg-half, avg+half


## 4. Verification
Visible hand-calculated event trace, deterministic zero-wait case, and analytical M/M/$c$ comparison with a predeclared tolerance. Analytical agreement is statistical, not an exact per-run identity.

In [ ]:
hand = simulate([0, 1, 1.5], [3, 2, 1], servers=1)
assert np.allclose(hand.waits, [0, 2, 3.5])
assert simulate([0, 10, 20], [1, 1, 1], servers=1).mean_wait == 0
assert random_trace(0.8, 1, 10, 7) == random_trace(0.8, 1, 10, 7)
print("Hand trace, zero-wait and seed reproducibility: PASS")

In [ ]:
rate, mean_service, servers = 0.8, 1.0, 2
samples = [replication(rate, mean_service, servers, 12000, seed=100+i)[0] for i in range(30)]
estimate, lower, upper = mean_ci(samples)
analytic = mmc_wait(rate, 1/mean_service, servers)
print(f"Simulated Wq = {estimate:.4f} min; 95% CI [{lower:.4f}, {upper:.4f}]")
print(f"M/M/c analytical Wq = {analytic:.4f} min")
assert abs(estimate - analytic) < 0.035, "M/M/c comparison outside predeclared 0.035-minute tolerance"

## 5. Experiments and results
Independent seeded replications; the confidence intervals quantify Monte Carlo uncertainty for the synthetic baseline, not uncertainty about real-world arrival assumptions.

In [ ]:
records=[]
for c in [1,2,3,4]:
    if rate >= c/mean_service: continue
    rep=[replication(rate, mean_service, c, 5000, 5000+c*100+i) for i in range(30)]
    w=mean_ci([v[0] for v in rep]); tail=mean_ci([v[1] for v in rep])
    records.append(dict(stations=c, mean_wait=w[0], ci_low=w[1], ci_high=w[2], p_wait_over_5=tail[0], utilization_theory=rate*mean_service/c))
results=pd.DataFrame(records)
display(results)
plt.errorbar(results.stations,results.mean_wait,yerr=[results.mean_wait-results.ci_low,results.ci_high-results.mean_wait],fmt='o-',capsize=4)
plt.xlabel('Inspection stations');plt.ylabel('Mean queue wait (minutes)');plt.title('Synthetic M/M/c scenario: 95% t intervals');plt.grid(alpha=.25);plt.show()

## 6. Real-data input calibration (X4)

Source: 4TU.ResearchData *Airport Security Checkpoint Data* (Rotterdam The Hague Airport). Its README defines **X** and **Y** as the start and end of a baggage check. The derived CSV contains only paired, valid positive inspection durations from session sheets listed in that README. Check `data/README.md` and `data/cleaning_audit.csv` for exclusions.

**Important limitation:** these observations calibrate the *service-time distribution*, not queue arrival rates, staffing or true waiting time. An M/G/c queue has no general simple Erlang-C wait formula. All arrivals in the following experiments are explicitly assumed scenarios.


In [ ]:
# This public source file must be committed to GitHub along with the notebook.
# GitHub-hosted Colab and local execution both use the same cleaned CSV.
from pathlib import Path
import urllib.request
local_path=Path('data/cleaned_baggage_check_durations.csv')
url='https://raw.githubusercontent.com/KawanRupasinghe/airport-baggage-queue/main/data/cleaned_baggage_check_durations.csv'
if local_path.exists():
    observed=pd.read_csv(local_path)
else:
    observed=pd.read_csv(io.BytesIO(urllib.request.urlopen(url,timeout=20).read()))
durations_minutes=observed.service_seconds.to_numpy(dtype=float)/60.0
assert len(durations_minutes)==162
assert np.all(np.isfinite(durations_minutes)) and np.all(durations_minutes>0)
print(f'Clean baggage checks: n={len(durations_minutes)}, mean={np.mean(durations_minutes)*60:.2f}s, median={np.median(durations_minutes)*60:.2f}s')
fig,ax=plt.subplots(figsize=(7,3));ax.hist(durations_minutes*60,bins=18,edgecolor='white');ax.set(xlabel='Baggage check service time (seconds)',ylabel='Observed frequency',title='Observed secondary baggage-check durations');plt.show()

In [ ]:
def empirical_replication(arrival_rate,stations,n,seed,warmup=500):
    rng=np.random.default_rng(seed)
    interarrival=rng.exponential(1/arrival_rate,size=n+warmup)
    arrival_times=np.cumsum(interarrival).tolist()
    services=rng.choice(durations_minutes,size=n+warmup,replace=True).tolist()
    result=simulate(arrival_times,services,servers=stations)
    post=np.asarray(result.waits[warmup:])
    return float(post.mean()),float(np.mean(post>2.0))

# Replication-level confidence intervals—not passenger-level pseudo-replication.
service_mean=float(durations_minutes.mean())
# Choose rates relative to estimated aggregate service capacity; these rates are assumptions.
records=[]
for load in [0.40,0.65,0.85]:
    for stations in [1,2,3]:
        arrival_rate=load*stations/service_mean
        estimates=[empirical_replication(arrival_rate,stations,2500,seed=1500+int(load*100)*100+stations*40+i) for i in range(24)]
        lo,lower,upper=mean_ci([x[0] for x in estimates])
        records.append(dict(target_utilisation=load,stations=stations,arrival_rate_per_minute=arrival_rate,mean_wait_min=lo,ci_low=lower,ci_high=upper,p_wait_gt_2_min=np.mean([x[1] for x in estimates])))
empirical_results=pd.DataFrame(records)
display(empirical_results.round(4))

### What these tests establish

The cleaned data provide a real observed *input distribution*; output waiting times remain predictions of a hypothetical shared-queue service system. Further work is needed for independent uncertainty studies, robust calibration sensitivity, measured Monte Carlo error scaling (X2) and systematic X5 results. These results must not be described as validated airport waiting-time measurements.


## 7. Extended verification: replication error scaling (X2)

**Independent reference:** the analytical M/M/1 mean queue wait is $W_q=\lambda/[\mu(\mu-\lambda)]$ for $0<\lambda<\mu$. We additionally test event traces and conservation. This extension measures the sampling error of a **mean of independent simulation replications**, rather than treating correlated passengers as independent.

Compute a pool of independent long-run M/M/1 estimates with deterministic seeds. Resample that pool to create repeated estimates based on different numbers of independent runs. The empirical *sampling* RMSE is measured against the fixed pool mean (not against a fictional airport observation); on log–log axes the expected sampling-error slope is approximately $-1/2$. This resampling check isolates sampling-error scaling, but finite-run bias remains a separate limitation.

In [ ]:
# Source provenance: AI-generated, human review required.
verify_rate, verify_service = 0.55, 1.0
pool=np.array([replication(verify_rate,verify_service,1,2200,seed=70000+j,warmup=600)[0] for j in range(240)])
mm1_exact=verify_rate/(1/verify_service*(1/verify_service-verify_rate))
print(f"Analytical M/M/1 Wq={mm1_exact:.4f} min; independent-run pool mean={pool.mean():.4f} min")
assert abs(pool.mean()-mm1_exact)<0.12, 'Long-run M/M/1 mismatch (predeclared tolerance)'
resample_rng=np.random.default_rng(98765)
levels=np.array([4,8,16,32,64,128])
rmse=[]
for m in levels:
    estimates=np.mean(resample_rng.choice(pool,size=(250,m),replace=True),axis=1)
    rmse.append(float(np.sqrt(np.mean((estimates-pool.mean())**2))))
rmse=np.asarray(rmse)
slope=float(np.polyfit(np.log(levels),np.log(rmse),1)[0])
print('Measured log-log slope:',round(slope,3),'(ideal independent-sampling slope: -0.5)')
fig,ax=plt.subplots(figsize=(6.8,4));ax.loglog(levels,rmse,'o-',label='Measured resampling RMSE');ax.loglog(levels,rmse[0]*np.sqrt(levels[0]/levels),'--',label='Reference N^-1/2');ax.set(xlabel='Number of independent replications',ylabel='RMSE relative to replication-pool mean',title='Measured Monte Carlo sampling-error scaling');ax.grid(True,which='both',alpha=.2);ax.legend();plt.show()
assert -0.85<slope<-0.2,'Measured slope inconsistent with inverse-square-root trend'


## 8. Systematic sensitivity analysis (X5)

We hold the **same assumed arrival rate** when comparing station counts (unlike a utilisation-normalised comparison, which changes demand with staffing). Scenarios vary three independently controlled factors: arrival rate $\lambda$, station count $c$, and a service-time multiplier $m$. Service durations come from the observed 162-case empirical distribution, multiplied by $m$. Scenarios do not represent measured demand at Rotterdam.

All reported intervals are 95% Student-$t$ intervals across **independent replications**. Each run discards its first 400 passengers as a warm-up; results describe long-run conditions for the tested range, not the time to clear a particular peak.

In [ ]:
def empirical_scaled_run(arrival_rate, stations, multiplier, n=1800, warmup=400, seed=0):
    rng=np.random.default_rng(seed)
    arrivals=np.cumsum(rng.exponential(1/arrival_rate,n+warmup))
    services=rng.choice(durations_minutes,n+warmup,replace=True)*multiplier
    result=simulate(arrivals.tolist(),services.tolist(),servers=stations,target_wait=2.0)
    waits=np.asarray(result.waits[warmup:])
    return float(waits.mean()), float((waits>2.0).mean())

sensitivity=[]
for arrival_rate in [0.4,0.65,0.85]:
  for c in [1,2,3]:
    for multiplier in [0.8,1.0,1.2]:
      rho=arrival_rate*service_mean*multiplier/c
      vals=[empirical_scaled_run(arrival_rate,c,multiplier,seed=900000+int(arrival_rate*100)*10000+c*1000+int(multiplier*10)*100+j) for j in range(16)]
      wq=mean_ci([x[0] for x in vals]);tail=mean_ci([x[1] for x in vals])
      sensitivity.append(dict(arrival_rate=arrival_rate,stations=c,service_multiplier=multiplier,nominal_utilisation=rho,mean_wait=wq[0],ci_low=wq[1],ci_high=wq[2],p_wait_gt2=tail[0]))
sensitivity_results=pd.DataFrame(sensitivity)
display(sensitivity_results.round(4))
base=sensitivity_results[sensitivity_results.service_multiplier==1.0].copy()
fig,ax=plt.subplots(figsize=(7.5,4))
for a,group in base.groupby('arrival_rate'):
  ax.plot(group.stations,group.mean_wait,'o-',label=f'{a:g} passengers/min')
ax.set(xlabel='Inspection stations',ylabel='Mean queue wait (minutes)',title='Sensitivity to demand at observed service durations',xticks=[1,2,3]);ax.legend(title='Arrival intensity');ax.grid(alpha=.2);plt.show()
# Model-conditional staffing decision for target E[Wq] <= 0.5 minutes at λ=0.65 per minute.
choice=base[base.arrival_rate==0.65].sort_values('stations')
qualifying=choice[choice.ci_high<=0.5]
print('Conservative model-conditional minimum stations (95% upper CI <= 0.5 min):',int(qualifying.stations.iloc[0]) if len(qualifying) else 'none of tested station counts')
print('Interpretation: not an operational recommendation for any actual airport; arrivals and topology assumed.')

### Sensitivity and uncertainty interpretation

Compare the three arrival rates at **fixed** staffing, then compare station counts at **fixed** arrival rate. The service multiplier probes the uncertainty inherent in carrying service observations across settings. The waiting-time target is a study assumption, not a screening authority requirement. Replication intervals quantify simulation noise *conditional on the model*; they do not cover uncertainty about arrival intensity, service sampling bias or the pooled queue assumption. The observed service sample is comparatively small and selected (only baggage checks actually recorded), so no airport-wide throughput or staffing claim is warranted.

## 9. Discussion and limitations
These are synthetic demonstrations, **not airport observations**. The source data must be audited before empirical fitting. Check whether timestamps measure screening starts, finishes, or checkpoint entry; arrival processes and selection into secondary inspection cannot be inferred without appropriate fields. No operational staffing recommendation is warranted from synthetic experiments alone. The current finite-customer warm-up discards early waiting observations but not queue-area counters.

## 10. References
- Gross, Shortle, Thompson and Harris, *Fundamentals of Queueing Theory* (bibliographic details to verify).
- Airport Security Checkpoint Data, 4TU.ResearchData (exact dataset version, DOI, licence and field descriptions to verify).

**References are placeholders pending source verification.**